Copyright **`(c)`** 2025 Giovanni Squillero `<giovanni.squillero@polito.it>`  
[`https://github.com/squillero/computational-intelligence`](https://github.com/squillero/computational-intelligence)  
Free under certain conditions — see the [`license`](https://github.com/squillero/computational-intelligence/blob/master/LICENSE.md) for details.  

In [2]:
from itertools import product, combinations
import numpy as np
import networkx as nx
from icecream import ic

In [3]:
def create_problem(
    size: int,
    *,
    density: float = 1.0,
    negative_values: bool = False,
    noise_level: float = 0.0,
    seed: int = 42,
) -> np.ndarray:
    """Problem generator for Lab3"""
    rng = np.random.default_rng(seed)
    # Genera N (size) punti casuali nel piano
    map = rng.random(size=(size, 2))
    # Inizializza la matrice dei pesi casuali
    problem = rng.random((size, size))
    if negative_values:
        problem = problem * 2 - 1
    problem *= noise_level
    for a, b in product(range(size), repeat=2):
        if rng.random() < density:
            # se l'arco esiste distanza euclidea + rumore (dell'inizializzazione)
            problem[a, b] += np.sqrt(
                np.square(map[a, 0] - map[b, 0]) + np.square(map[a, 1] - map[b, 1])
            )
        else:
            # se l'arco non esiste peso infinito
            problem[a, b] = np.inf
    np.fill_diagonal(problem, 0)
    return (problem * 1_000).round()


# Crea N punti in un piano 2D.
#Calcola le loro distanze euclidee.
#Usa queste distanze come pesi degli archi del grafo.
#Con density < 1, alcuni archi vengono rimossi (posti a inf → nessun collegamento).

#Aggiunge eventualmente rumore e valori negativi.

In [62]:
problem = create_problem(20, density=0.15, noise_level=10, negative_values=True)
problem

array([[ 0.000e+00,        inf,        inf,        inf,        inf,
               inf,        inf,        inf,  4.084e+03,        inf,
        -5.916e+03,        inf,        inf,        inf,  4.769e+03,
               inf,        inf,        inf,        inf,        inf],
       [       inf,  0.000e+00,        inf,        inf,        inf,
               inf,        inf,        inf, -8.681e+03,        inf,
               inf,        inf,  7.577e+03, -4.559e+03,        inf,
               inf,        inf,        inf,        inf,        inf],
       [       inf,        inf,  0.000e+00,        inf, -9.019e+03,
               inf,        inf,        inf,        inf,        inf,
               inf,        inf,        inf,        inf,        inf,
               inf,        inf,        inf,        inf,        inf],
       [       inf,        inf,        inf,  0.000e+00,        inf,
               inf,        inf,        inf,        inf,  8.220e+03,
        -4.530e+02,        inf, -3.289e+03,  

In [49]:
masked = np.ma.masked_array(problem, mask=np.isinf(problem))
G = nx.from_numpy_array(masked, create_using=nx.DiGraph)

In [51]:
for s, d in combinations(range(problem.shape[0]), 2):
    try:
        # path = nx.shortest_path(G, s, d, weight='weight')
        path = nx.bellman_ford_path(G, s, d, weight='weight')
        cost = cost = nx.path_weight(G, path, weight='weight')
    except nx.NetworkXNoPath:
        # Nodes are not connected
        path = None
        cost = np.inf
    except nx.NetworkXUnbounded:
        # Negative cycle detected
        path = None
        cost = -np.inf
    path_2, cost_2 = best_fit(problem, s, d) 
    if path != path_2 or cost != cost_2:
        ic("Mismatch detected!")
        ic(f"Nodes: {s} -> {d}")
        ic(f"NetworkX path: {path}, cost: {cost}")
        ic(f"Best-fit path: {path_2}, cost: {cost_2}")
None

ic| 'Mismatch detected!'
ic| f"Nodes: {s} -> {d}": 'Nodes: 0 -> 4'
ic| f"NetworkX path: {path}, cost: {cost}": 'NetworkX path: [0, 10, 2, 4], cost: 6074.0'
ic| f"Best-fit path: {path_2}, cost: {cost_2}": 'Best-fit path: [0, 14, 4], cost: 16438.0'
ic| 'Mismatch detected!'
ic| f"Nodes: {s} -> {d}": 'Nodes: 0 -> 7'
ic| f"NetworkX path: {path}, cost: {cost}": 'NetworkX path: [0, 10, 9, 7], cost: 12740.0'
ic| f"Best-fit path: {path_2}, cost: {cost_2}": 'Best-fit path: [0, 14, 7], cost: 13585.0'
ic| 'Mismatch detected!'
ic| f"Nodes: {s} -> {d}": 'Nodes: 1 -> 3'
ic| f"NetworkX path: {path}, cost: {cost}": 'NetworkX path: [1, 8, 7, 14, 11, 3], cost: 14599.0'
ic| f"Best-fit path: {path_2}, cost: {cost_2}": 'Best-fit path: [1, 8, 18, 11, 3], cost: 19992.0'
ic| 'Mismatch detected!'
ic| f"Nodes: {s} -> {d}": 'Nodes: 1 -> 11'
ic| f"NetworkX path: {path}, cost: {cost}": 'NetworkX path: [1, 8, 7, 14, 11], cost: 12925.0'
ic| f"Best-fit path: {path_2}, cost: {cost_2}": 'Best-fit path: [1, 8, 18, 11], c

In [59]:
import numpy as np

def best_fit(graph: np.ndarray, start: int, goal: int) -> tuple[list[int], float]:
    
    size = graph.shape[0]
    # ( nodo, costo accumulato )
    
    #esclude percorsi circolari
    # mantiene per ogni percorso i nodi visitati
    feasible_paths=[]
    frontier=[]
    path_id = 0
    for i in range(size):
        if i != start and graph[start, i] != np.inf:
            frontier.append((path_id, i, graph[start, i]))
            feasible_paths.append([start, i])
            path_id += 1

    
    #print("frontiera iniziale:", frontier)
    #print("path possibili iniziali: ", feasible_paths)
    goal_path=[]    

    while frontier:
        #print ("frontiera:", frontier)
        path_id, current_node, current_cost = frontier.pop(0)
        #print ("estraggo:", path_id, current_node, current_cost)
        if current_node == goal:
            
            path= feasible_paths[path_id]
            #print("goal reached", path)
            if path not in [p[0] for p in goal_path]:
             goal_path.append((path, current_cost))
             rate= len(goal_path)/len(feasible_paths)
             #ic(path_id, path, current_cost,rate )
            # if rate < 0.001:
              #   break
            continue
            

        for neighbor in range(size):
            weight = graph[current_node, neighbor]
            if weight != np.inf and neighbor not in feasible_paths[path_id]:
                current_path= feasible_paths[path_id]
                new_path=current_path + [neighbor]
                ## evita percorsi uguali
                if new_path not in [p for p in feasible_paths]:
                    ##add new path
                    new_path_id= len(feasible_paths)
                    feasible_paths.append(new_path)

                    #print(new_path_id, feasible_paths[new_path_id])
                    frontier.append((new_path_id, neighbor, current_cost + weight))
                #else:
                    #print("ignoro nodo:", neighbor, " path:", current_path)
            #else:
                #print("ignoro nodo:", neighbor, "weight:", weight)
                
        # ordina la coda in base al costo accumulato (crescente)
        frontier.sort(key=lambda x: x[1])  # best-first
        
    if goal_path:
        # ordino in ordine crescente di lunghezza e a parità di lunghezza di costo
        # potrai far partire un algoritmo genetico qua perchè i percorsi sono validi posso modificarli minimamente per trovarne di migliori
        return min(goal_path, key=lambda x:  x[1])
       
    return None, np.inf


s, d = 0, 9
path, cost = best_fit(problem, s, d)
print( "start:", s, " end:", d, " path:", path, " cost:", cost)

start: 0  end: 9  path: [0, 7, 2, 9]  cost: -15377.0


In [63]:
## aggiustarlo per i cicli negativi
def distances_bellman_ford(problem: np.ndarray, start: int) -> tuple[list[int], list[float]]:
    """Bellman-Ford algorithm to find shortest paths from a single source."""
    size = problem.shape[0]
    distances = [np.inf] * size
    predecessors = [None] * size
    distances[start] = 0

    for _ in range(size - 1):
        for u in range(size):
            for v in range(size):
                #per ogni arco estrae il peso
                weight = problem[u, v]
                if weight != np.inf and distances[u] + weight < distances[v] :
                    #se la distanza da sorgente a u più w è minore della distanza corrente da sorgente a v, allora aggiorna la distanza di v
                    
                    # aggiorna solamente se non crea un ciclo negativo
                    visited_nodes=[v]
                    is_negative_cycle= False
                    current_node=u
                    for _ in range(size):
                        current_node= predecessors[current_node]
                        if current_node is None:
                            break
                        if current_node in visited_nodes:
                            is_negative_cycle= True
                            break
                        visited_nodes.append(current_node)
                         

                    if not is_negative_cycle:
                        predecessors[v] = u
                        distances[v] = distances[u] + weight

    return predecessors, distances
   


def shorthest_path_bellman_ford(predecessors, distances, end: int) -> tuple[list[int], float]:
    path = []
    current_node = end 
    while current_node is not None:
        path.insert(0, current_node)
        current_node = predecessors[current_node]
    return path, distances[end]


predecessors, distances = distances_bellman_ford(problem, 0)
path, cost = shorthest_path_bellman_ford(predecessors, distances, 9)
print(path, cost)

[0, 10, 2, 4, 14, 11, 3, 18, 5, 7, 9] -24953.0


In [64]:
def path_cost(path, problem):
    total_cost=0
    for i in range(len(path)-1):
        u= path[i]
        v= path[i+1]
        total_cost += problem[u,v]
    return total_cost

path_cost(path, problem)

np.float64(-24953.0)

In [65]:

for s, d in combinations(range(problem.shape[0]), 2):
    predecessors, distances = distances_bellman_ford(problem, s)
    path, cost = shorthest_path_bellman_ford(predecessors, distances, d)
    print("start:", s, " end:", d, " cost:", cost, "\t path:", path  )
    if problem.size <= 20:
    # confronto con best fit
        path_2, cost_2 = best_fit(problem, s, d)
        if cost_2 != cost:
            print("Mismatch detected!")
            print(f"Nodes: {s} -> {d}")
            print(f"Bellman-Ford path: {path}, cost: {cost}")
            print(f"Best-fit path: {path_2}, cost: {cost_2}")
None

start: 0  end: 1  cost: -9049.0 	 path: [0, 10, 2, 4, 14, 11, 3, 18, 5, 7, 9, 15, 1]
start: 0  end: 2  cost: -10354.0 	 path: [0, 10, 2]
start: 0  end: 3  cost: -29317.0 	 path: [0, 10, 2, 4, 14, 11, 3]
start: 0  end: 4  cost: -19373.0 	 path: [0, 10, 2, 4]
start: 0  end: 5  cost: -21877.0 	 path: [0, 10, 2, 4, 14, 11, 3, 18, 5]
start: 0  end: 6  cost: -3372.0 	 path: [0, 10, 6]
start: 0  end: 7  cost: -20760.0 	 path: [0, 10, 2, 4, 14, 11, 3, 18, 5, 7]
start: 0  end: 8  cost: -17730.0 	 path: [0, 10, 2, 4, 14, 11, 3, 18, 5, 7, 9, 15, 1, 8]
start: 0  end: 9  cost: -24953.0 	 path: [0, 10, 2, 4, 14, 11, 3, 18, 5, 7, 9]
start: 0  end: 10  cost: -5916.0 	 path: [0, 10]
start: 0  end: 11  cost: -22430.0 	 path: [0, 10, 2, 4, 14, 11]
start: 0  end: 12  cost: -32606.0 	 path: [0, 10, 2, 4, 14, 11, 3, 12]
start: 0  end: 13  cost: -13608.0 	 path: [0, 10, 2, 4, 14, 11, 3, 18, 5, 7, 9, 15, 1, 13]
start: 0  end: 14  cost: -17321.0 	 path: [0, 10, 2, 4, 14]
start: 0  end: 15  cost: -18520.0 	 pat